# A8 calibration and locked evaluation
Run calibration before the locked test is copied into the runtime. All logic is in `sipature_ml.evaluation`; this notebook only orchestrates the two immutable phases.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!test -d /content/hackathon/.git || git clone REPLACE_REPOSITORY_URL /content/hackathon
!git -C /content/hackathon pull --ff-only
%cd /content/hackathon/ml
!python -m pip uninstall -y torchvision
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .

## Phase 1: validation only
Set controlled Drive source paths below. Copy only the manifest and validation file for Phase 1; the locked test remains in Drive. The model run must be the full A7 Drive run.

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

from sipature_ml.evaluation import run_calibration

SPLIT_DIR = Path('/content/a8-splits')
DRIVE_SPLIT_DIR = Path('/content/drive/MyDrive/SIPATURE/frozen-splits')
MANIFEST_SOURCE = DRIVE_SPLIT_DIR / 'split_manifest_silver_v1.json'
VALIDATION_SOURCE = DRIVE_SPLIT_DIR / 'validation_silver_v1.jsonl'
LOCKED_TEST_SOURCE = DRIVE_SPLIT_DIR / 'test_silver_v1.jsonl'
MODEL_RUN_DIR = Path('/content/drive/MyDrive/SIPATURE/runs/REPLACE_A7_RUN_ID')
CALIBRATION_DIR = Path('/content/drive/MyDrive/SIPATURE/calibration/REPLACE_CALIBRATION_ID')
SPLIT_DIR.mkdir(exist_ok=False)
shutil.copy2(MANIFEST_SOURCE, SPLIT_DIR / MANIFEST_SOURCE.name)
shutil.copy2(VALIDATION_SOURCE, SPLIT_DIR / VALIDATION_SOURCE.name)
manifest = json.loads((SPLIT_DIR / MANIFEST_SOURCE.name).read_text())
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
assert sha256(SPLIT_DIR / VALIDATION_SOURCE.name) == manifest['outputs']['validation']['sha256']
print({'manifest_sha256': sha256(SPLIT_DIR / MANIFEST_SOURCE.name), 'validation_sha256': sha256(SPLIT_DIR / VALIDATION_SOURCE.name)})
calibration = run_calibration(SPLIT_DIR, MODEL_RUN_DIR, CALIBRATION_DIR)
assert calibration['test_read'] is False
calibration

## Mandatory pause
Stop here. Inspect `calibration.json` and `manifest.json`, record their hashes, and freeze the directory. Do not continue until a human confirms the artifact is accepted. Only after that confirmation may the locked test file be copied from controlled Drive storage to `SPLIT_DIR`; do not use browser upload or inspect its contents.

In [ ]:
CONFIRMATION_PHRASE = 'I AUTHORIZE ONE LOCKED TEST ACCESS'
confirmation = input(f'Type exactly: {CONFIRMATION_PHRASE}\n')
assert confirmation == CONFIRMATION_PHRASE, 'Locked test access is not authorized'

## Phase 2: one locked-test pass
After the pause, copy the hash-locked test from controlled Drive into `SPLIT_DIR`, then execute this cell once. Use one predeclared output ID. Never choose a fresh ID after any attempted locked-test access; preserve `evaluation-state.json` and document an incident.

In [ ]:
from sipature_ml.evaluation import run_locked_test_evaluation

assert confirmation == CONFIRMATION_PHRASE
shutil.copy2(LOCKED_TEST_SOURCE, SPLIT_DIR / LOCKED_TEST_SOURCE.name)
EVALUATION_DIR = Path('/content/drive/MyDrive/SIPATURE/evaluation/REPLACE_EVALUATION_ID')
BASELINE_METRICS_DIR = Path('/content/hackathon/ml/artifacts/metrics')
metrics = run_locked_test_evaluation(
    SPLIT_DIR, MODEL_RUN_DIR, CALIBRATION_DIR, EVALUATION_DIR, BASELINE_METRICS_DIR
)
assert metrics['test_inference_passes'] == 1
metrics